# LeetCode 2717 — Semi-Ordered Permutation bằng Python

Link đề: https://leetcode.com/problems/semi-ordered-permutation/ · Độ khó: **Easy** · Chủ đề: Array, Simulation

Skeleton LeetCode:

```python
class Solution:
    def semiOrderedPermutation(self, nums: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho một **hoán vị** của `1..n`; mỗi thao tác **đổi chỗ hai phần tử kề nhau**; tìm số thao tác **ít nhất** để số **đầu tiên là `1`** và số **cuối cùng là `n`** (các số ở giữa tuỳ ý).
2. Biết **hoán vị (permutation)** là gì và tại sao mỗi giá trị `1..n` xuất hiện đúng một lần ⇒ vị trí của `1` và `n` là **duy nhất**.
3. Nắm cú pháp: `list.index`, hoán đổi `a[i], a[i+1] = a[i+1], a[i]`, `enumerate`, `pop` / `insert`, và `True`/`False` dùng như `1`/`0` trong phép cộng.
4. Phát hiện công thức: `vị_trí_của_1 + (n − 1 − vị_trí_của_n)`, **trừ `1` nếu `1` nằm sau `n`** (hai số này "đổi chỗ chung" một lần).
5. Viết 4 cách: mô phỏng từng lần đổi chỗ (baseline), công thức với hai `index`, công thức với một lượt duyệt, và mô phỏng bằng "nhấc phần tử ra, đặt vào đầu/cuối".
6. Kiểm chứng bằng **BFS vét cạn** (tìm đường ngắn nhất trong không gian hoán vị) và test ngẫu nhiên có seed.
7. Tránh các lỗi quen thuộc: quên trừ `1` khi chồng chéo, tính sai khoảng cách tới cuối, sắp xếp cả mảng.

## 2. Đề bài & ví dụ

Cho mảng `nums` đánh chỉ số từ `0`, là một **hoán vị** của `n` số nguyên `1, 2, ..., n` (mỗi số từ `1` đến `n` xuất hiện **đúng một lần**, `n = len(nums)`).

Một hoán vị gọi là **semi-ordered** nếu **số đầu tiên bằng `1`** và **số cuối cùng bằng `n`**.

Bạn được thực hiện thao tác sau bao nhiêu lần tuỳ ý: **đổi chỗ hai phần tử kề nhau** (hai phần tử ở chỉ số `i` và `i + 1`).

Trả về **số thao tác tối thiểu** để biến `nums` thành hoán vị semi-ordered.

| Ví dụ | nums | Output | Giải thích |
|---|---|---|---|
| 1 | `[2,1,4,3]` | `2` | đổi `(2,1)` → `[1,2,4,3]`; đổi `(4,3)` → `[1,2,3,4]` |
| 2 | `[2,4,1,3]` | `3` | đổi `(4,1)` → `[2,1,4,3]`; đổi `(2,1)` → `[1,2,4,3]`; đổi `(4,3)` → `[1,2,3,4]` |
| 3 | `[1,3,4,2,5]` | `0` | đã semi-ordered (đầu `1`, cuối `5`) |

Constraints: `2 <= nums.length <= 50`; `1 <= nums[i] <= 50`; `nums` là một hoán vị của `n` số nguyên (tức gồm đúng `1..n`).

In [1]:
nums = [2, 4, 1, 3]
a = nums[:]
for i in (1, 0, 2):             # thực hiện ba lần đổi chỗ như ví dụ 2: (4,1), (2,1), (4,3) -> chỉ số 1, 0, 2
    a[i], a[i + 1] = a[i + 1], a[i]
    print("đổi chỗ chỉ số", i, "và", i + 1, "->", a)
print("đầu =", a[0], "| cuối =", a[-1], "| n =", len(a))

đổi chỗ chỉ số 1 và 2 -> [2, 1, 4, 3]
đổi chỗ chỉ số 0 và 1 -> [1, 2, 4, 3]
đổi chỗ chỉ số 2 và 3 -> [1, 2, 3, 4]
đầu = 1 | cuối = 4 | n = 4


## 3. Kiến thức nền

### 3.1. Hoán vị và vị trí duy nhất

Hoán vị của `1..n` là danh sách chứa **mỗi số đúng một lần**. Nhờ vậy `1` xuất hiện **một** chỗ, `n` xuất hiện **một** chỗ. Tìm vị trí bằng `nums.index(giá_trị)` (trả về chỉ số xuất hiện **đầu tiên**; ném `ValueError` nếu không có).

In [2]:
nums = [2, 4, 1, 3]
n = len(nums)
print(nums.index(1), nums.index(n))        # 2 1 : số 1 ở chỉ số 2, số 4 ở chỉ số 1
print(sorted(nums) == list(range(1, n + 1)))   # True: đúng là hoán vị của 1..n

2 1
True


### 3.2. Hoán đổi hai phần tử kề nhau

Python hoán đổi gọn: `a[i], a[i + 1] = a[i + 1], a[i]` (vế phải được tính xong rồi mới gán vế trái). Mỗi lần như vậy là **một** thao tác của đề.

In [3]:
a = [2, 1, 4, 3]
a[0], a[1] = a[1], a[0]
print(a)                    # [1, 2, 4, 3]
a[2], a[3] = a[3], a[2]
print(a)                    # [1, 2, 3, 4]

[1, 2, 4, 3]
[1, 2, 3, 4]


### 3.3. Khoảng cách để "đi" tới đầu/cuối bằng đổi chỗ kề nhau

Muốn đưa phần tử ở chỉ số `p` về **đầu** (chỉ số `0`) bằng cách đổi với phần tử kề bên trái, cần đúng `p` lần đổi (mỗi lần lùi một ô). Muốn đưa về **cuối** (chỉ số `n - 1`) cần `n - 1 - p` lần.

In [4]:
a = [2, 3, 1, 4]       # số 1 ở chỉ số 2
p = a.index(1)
cnt = 0
while p > 0:
    a[p - 1], a[p] = a[p], a[p - 1]
    p -= 1
    cnt += 1
print(a, "số lần đổi:", cnt)       # [1, 2, 3, 4] 2

[1, 2, 3, 4] số lần đổi: 2


### 3.4. `True` / `False` là `1` / `0`

Trong Python, `True == 1` và `False == 0`, nên `(a > b)` có thể **cộng/trừ** như số: `5 - (3 > 2)` là `4`. Gọn nhưng nên viết rõ `1 if ... else 0` khi mới học.

In [5]:
print(True + True, 5 - (3 > 2), 5 - (2 > 3))      # 2 4 5
i1, i_n = 2, 1
print(1 if i1 > i_n else 0, int(i1 > i_n))         # 1 1

2 4 5
1 1


### 3.5. `enumerate`, `pop(i)`, `insert(i, x)`, `append`

- `for i, x in enumerate(a)` duyệt cùng lúc chỉ số và giá trị.
- `a.pop(i)` lấy phần tử ở `i` ra (các phần tử sau dịch trái), `a.insert(0, x)` chèn `x` vào đầu, `a.append(x)` đặt vào cuối.

In [6]:
a = [2, 4, 1, 3]
for i, x in enumerate(a):
    print(i, x, end=" | ")
print()
x = a.pop(2)               # lấy số 1 ra
a.insert(0, x)             # đặt vào đầu
print(a)                   # [1, 2, 4, 3]
y = a.pop(2)               # lấy số 4 ra
a.append(y)                # đặt vào cuối
print(a)                   # [1, 2, 3, 4]

0 2 | 1 4 | 2 1 | 3 3 | 
[1, 2, 4, 3]
[1, 2, 3, 4]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ta chỉ cần quan tâm **hai số: `1` và `n`**. Các số khác nằm đâu cũng được ở bước cuối. Hãy nghĩ như đưa hai "khách VIP" tới hai đầu hàng:

- `1` ở chỉ số `p1` cần đi **lùi `p1` ô** về đầu.
- `n` ở chỉ số `pn` cần đi **tiến `n - 1 - pn` ô** về cuối.

Tạm cộng hai quãng đường: `p1 + (n - 1 - pn)`.

**Nhưng** nếu `1` đứng **sau** `n` (`p1 > pn`) thì hai người phải **đi ngang qua nhau**; khi `1` lùi qua `n`, một lần đổi chỗ đó **vừa đẩy `1` lùi, vừa đẩy `n` tiến** ⇒ tính hai lần cho một thao tác. Phải **trừ `1`**.

Tính tay `[2,4,1,3]`: `p1 = 2`, `pn = 1`, `n = 4`: `2 + (4 - 1 - 1) = 4`; vì `p1 > pn` nên trừ 1 ⇒ `3` ✓.

In [7]:
def tinh_tay(nums):
    n = len(nums)
    p1, pn = nums.index(1), nums.index(n)
    tong = p1 + (n - 1 - pn)
    return tong - 1 if p1 > pn else tong

print(tinh_tay([2, 1, 4, 3]), tinh_tay([2, 4, 1, 3]), tinh_tay([1, 3, 4, 2, 5]))   # 2 3 0

2 3 0


## 5. Cách 1 — Brute Force / baseline: mô phỏng từng lần đổi chỗ

**Ý tưởng:** làm đúng như đề: (1) tìm `1`, rồi **đổi chỗ liên tục với phần tử bên trái** cho tới khi `1` ở đầu — đếm số lần; (2) tìm `n`, **đổi chỗ liên tục với phần tử bên phải** cho tới khi `n` ở cuối — đếm số lần. Cộng hai số đếm.

Chạy trên **bản sao** `a = nums[:]` để không làm hỏng dữ liệu gốc.

In [8]:
from typing import List

def semi_ordered_simulate(nums: List[int]) -> int:
    a = nums[:]
    n = len(a)
    ops = 0
    p = a.index(1)
    while p > 0:                          # đưa 1 về đầu
        a[p - 1], a[p] = a[p], a[p - 1]
        p -= 1
        ops += 1
    q = a.index(n)
    while q < n - 1:                      # đưa n về cuối
        a[q], a[q + 1] = a[q + 1], a[q]
        q += 1
        ops += 1
    assert a[0] == 1 and a[-1] == n
    return ops

print(semi_ordered_simulate([2, 1, 4, 3]))      # 2
print(semi_ordered_simulate([2, 4, 1, 3]))      # 3
print(semi_ordered_simulate([1, 3, 4, 2, 5]))   # 0

2
3
0


**Độ phức tạp:** mỗi vòng đổi chỗ tốn `O(1)`, tối đa `n - 1` lần cho mỗi số ⇒ Time `O(n)`; thêm `index` `O(n)`. Space `O(n)` cho bản sao.

**Nhận xét:** thực ra không cần thật sự đổi chỗ: chỉ cần biết **số lần**, mà số lần chính là khoảng cách `p` hoặc `n - 1 - q`... **Nhưng chú ý** `q` ở đây đã **dịch chuyển** khi `1` đi qua `n` — chi tiết đó nằm ở thủ thuật "trừ 1" (mục 6).

## 6. Ý tưởng thuật toán / bước đột phá

Trong mô phỏng ở Cách 1, ta thấy hai trường hợp:

- **`1` đứng trước `n` (`p1 < pn`):** khi `1` lùi về đầu, nó chỉ đi qua các phần tử **bên trái** `p1`; `n` đứng ở bên phải nên **không bị động tới**. Sau đó `n` vẫn ở `pn`, cần `n - 1 - pn` bước. Tổng `p1 + (n - 1 - pn)`.
- **`1` đứng sau `n` (`p1 > pn`):** khi `1` lùi về đầu, nó phải đi **qua `n`**; lần đổi chỗ đó làm `n` **tiến thêm một ô** (từ `pn` lên `pn + 1`). Khi đó `n` đã tiến sẵn một bước, chỉ còn `n - 1 - (pn + 1)` bước nữa. Tổng `p1 + (n - 1 - pn) - 1`.

Gộp lại: **`p1 + (n - 1 - pn) - (1 nếu p1 > pn, ngược lại 0)`** — không cần mô phỏng!

In [9]:
nums = [2, 4, 1, 3]
n = len(nums)
p1, pn = nums.index(1), nums.index(n)
print("p1 =", p1, "pn =", pn, "n =", n)
print("trường hợp p1 > pn:", p1 > pn)
print("đáp án =", p1 + (n - 1 - pn) - (1 if p1 > pn else 0))

p1 = 2 pn = 1 n = 4
trường hợp p1 > pn: True
đáp án = 3


## 7. Các cách tiếp theo

Ngoài Cách 1 (mô phỏng), ta xem 3 cách: (2) công thức với hai `index`, (3) công thức với **một lượt** duyệt, (4) mô phỏng "nhấc phần tử ra rồi đặt vào đầu/cuối" bằng `pop`/`insert`/`append`.

### Cách 2 — Công thức với `index`

Giải thích từng đoạn:

- `n = len(nums)`.
- `i = nums.index(1)`, `j = nums.index(n)`: vị trí của `1` và `n`.
- `ans = i + (n - 1 - j)`: khoảng cách tới đầu và tới cuối.
- `if i > j: ans -= 1`: trừ phần chồng chéo.

In [10]:
def semi_ordered_index(nums: List[int]) -> int:
    n = len(nums)
    i = nums.index(1)
    j = nums.index(n)
    ans = i + (n - 1 - j)
    if i > j:
        ans -= 1
    return ans

print(semi_ordered_index([2, 1, 4, 3]), semi_ordered_index([2, 4, 1, 3]), semi_ordered_index([1, 3, 4, 2, 5]))

2 3 0


In [11]:
from typing import List

class SolutionIndex:
    def semiOrderedPermutation(self, nums: List[int]) -> int:
        n = len(nums)
        i = nums.index(1)
        j = nums.index(n)
        ans = i + (n - 1 - j)
        if i > j:
            ans -= 1
        return ans

print(SolutionIndex().semiOrderedPermutation([2, 4, 1, 3]))

3


**Độ phức tạp:** hai lần `index` mỗi lần `O(n)` ⇒ Time `O(n)`, Space `O(1)`.

### Cách 3 — Một lượt duyệt `enumerate`

Hai lần `index` duyệt mảng hai lần. Ta có thể tìm **cả hai vị trí trong một vòng** `for i, x in enumerate(nums)`: nếu `x == 1` ghi `i1 = i`; nếu `x == n` ghi `in_ = i`. Công thức giữ nguyên.

Tên biến: `in` là **từ khoá** của Python nên không đặt tên `in`; ta dùng `pn` (position of n).

In [12]:
def semi_ordered_one_pass(nums: List[int]) -> int:
    n = len(nums)
    p1 = pn = -1
    for i, x in enumerate(nums):
        if x == 1:
            p1 = i
        elif x == n:
            pn = i
    return p1 + (n - 1 - pn) - (1 if p1 > pn else 0)

print(semi_ordered_one_pass([2, 1, 4, 3]), semi_ordered_one_pass([2, 4, 1, 3]), semi_ordered_one_pass([1, 3, 4, 2, 5]))

2 3 0


In [13]:
from typing import List

class SolutionOnePass:
    def semiOrderedPermutation(self, nums: List[int]) -> int:
        n = len(nums)
        p1 = pn = -1
        for i, x in enumerate(nums):
            if x == 1:
                p1 = i
            elif x == n:
                pn = i
        return p1 + (n - 1 - pn) - (1 if p1 > pn else 0)

print(SolutionOnePass().semiOrderedPermutation([2, 1, 4, 3]))

2


**Độ phức tạp:** một lượt `O(n)`, Space `O(1)`. (Với `n ≥ 2`, `1` và `n` là hai giá trị **khác nhau**, nên `elif` không bỏ sót; nếu `n = 1` thì `1 == n` — nhưng đề có `n ≥ 2`.)

### Cách 4 — "Nhấc ra, đặt vào" bằng `pop` / `insert` / `append`

Một góc nhìn khác: một phần tử có thể "chạy" qua `k` phần tử bằng đúng `k` lần đổi chỗ kề nhau — giống như **nhấc nó ra rồi đặt vào vị trí mới**. Chi phí di chuyển = số phần tử mà nó đi qua = khoảng cách chỉ số.

1. `i = a.index(1)`; `a.insert(0, a.pop(i))`; tốn `i`.
2. `j = a.index(n)` (**tính lại sau bước 1**, vì chỉ số có thể đã đổi); `a.append(a.pop(j))`; tốn `n - 1 - j`.

Cách này cho thấy rõ **vì sao phải trừ 1**: sau bước 1, nếu `n` đứng trước `1` thì chỉ số `j` của `n` **tăng thêm 1** nên bước 2 rẻ hơn 1.

In [14]:
def semi_ordered_pop_insert(nums: List[int]) -> int:
    a = nums[:]
    n = len(a)
    ops = 0
    i = a.index(1)
    ops += i
    a.insert(0, a.pop(i))
    j = a.index(n)                # tìm lại vì chỉ số đã đổi
    ops += n - 1 - j
    a.append(a.pop(j))
    assert a[0] == 1 and a[-1] == n
    return ops

print(semi_ordered_pop_insert([2, 1, 4, 3]), semi_ordered_pop_insert([2, 4, 1, 3]), semi_ordered_pop_insert([1, 3, 4, 2, 5]))

2 3 0


In [15]:
from typing import List

class SolutionPopInsert:
    def semiOrderedPermutation(self, nums: List[int]) -> int:
        a = nums[:]
        n = len(a)
        i = a.index(1)
        ops = i
        a.insert(0, a.pop(i))
        j = a.index(n)
        ops += n - 1 - j
        return ops

print(SolutionPopInsert().semiOrderedPermutation([2, 4, 1, 3]))

3


**Độ phức tạp:** `index`, `pop`, `insert`, `append` đều `O(n)` nhưng chỉ gọi hằng số lần ⇒ Time `O(n)`, Space `O(n)` (bản sao `a`).

## 8. TRACE từng bước

Trace Cách 1 (mô phỏng) với `nums = [2, 4, 1, 3]` (`n = 4`):

| bước | thao tác | mảng sau | tổng |
|---|---|---|---|
| (đầu) | `1` ở chỉ số 2 | `[2,4,1,3]` | 0 |
| 1 | đổi `4` và `1` (chỉ số 1, 2) — **`1` lùi, `4` tiến** | `[2,1,4,3]` | 1 |
| 2 | đổi `2` và `1` (chỉ số 0, 1) — `1` về đầu | `[1,2,4,3]` | 2 |
| 3 | `4` ở chỉ số 2; đổi `4` và `3` (chỉ số 2, 3) — `4` về cuối | `[1,2,3,4]` | 3 |

Bước 1 là lần đổi chỗ **chung** của `1` và `n`: ta vừa tính nó như một bước của `1`, vừa đã làm `4` tiến một ô. Do đó `4` chỉ còn cần `1` bước thay vì `2`.

In [16]:
def trace_simulate(nums):
    a = nums[:]
    n = len(a)
    ops = 0
    print("bắt đầu:", a)
    p = a.index(1)
    while p > 0:
        a[p - 1], a[p] = a[p], a[p - 1]
        p -= 1
        ops += 1
        print(f"  đưa 1 về đầu: đổi chỉ số {p} và {p + 1} -> {a}  (ops={ops})")
    q = a.index(n)
    print(f"  bây giờ {n} ở chỉ số {q}")
    while q < n - 1:
        a[q], a[q + 1] = a[q + 1], a[q]
        q += 1
        ops += 1
        print(f"  đưa {n} về cuối: đổi chỉ số {q - 1} và {q} -> {a}  (ops={ops})")
    print("kết quả:", a, "| số thao tác:", ops)
    return ops

trace_simulate([2, 4, 1, 3])

bắt đầu: [2, 4, 1, 3]
  đưa 1 về đầu: đổi chỉ số 1 và 2 -> [2, 1, 4, 3]  (ops=1)
  đưa 1 về đầu: đổi chỉ số 0 và 1 -> [1, 2, 4, 3]  (ops=2)
  bây giờ 4 ở chỉ số 2
  đưa 4 về cuối: đổi chỉ số 2 và 3 -> [1, 2, 3, 4]  (ops=3)
kết quả: [1, 2, 3, 4] | số thao tác: 3


3

In [17]:
# Trace công thức: so sánh hai trường hợp p1 < pn và p1 > pn
for nums in ([2, 1, 4, 3], [2, 4, 1, 3]):
    n = len(nums)
    p1, pn = nums.index(1), nums.index(n)
    sub = 1 if p1 > pn else 0
    print(f"{nums}: p1={p1}, pn={pn}, n-1-pn={n - 1 - pn}, p1 > pn? {p1 > pn} -> trừ {sub} => {p1 + (n - 1 - pn) - sub}")

[2, 1, 4, 3]: p1=1, pn=2, n-1-pn=1, p1 > pn? False -> trừ 0 => 2
[2, 4, 1, 3]: p1=2, pn=1, n-1-pn=2, p1 > pn? True -> trừ 1 => 3


## 9. Code nộp LeetCode

Dùng Cách 2 (công thức hai `index`), đúng class `Solution` và method `semiOrderedPermutation`.

In [18]:
from typing import List

class Solution:
    def semiOrderedPermutation(self, nums: List[int]) -> int:
        n = len(nums)
        i = nums.index(1)
        j = nums.index(n)
        ans = i + (n - 1 - j)
        if i > j:
            ans -= 1
        return ans

In [19]:
print(Solution().semiOrderedPermutation([2, 1, 4, 3]))      # 2
print(Solution().semiOrderedPermutation([2, 4, 1, 3]))      # 3
print(Solution().semiOrderedPermutation([1, 3, 4, 2, 5]))   # 0

2
3
0


## 10. Vì sao thuật toán đúng

Gọi `p1`, `pn` là chỉ số của `1` và `n`. Ta chứng minh **cận dưới** và **cận trên** bằng nhau.

**Cận trên (xây được đúng số bước ấy).**

- Nếu `p1 < pn`: đưa `1` về đầu bằng `p1` lần đổi chỗ với phần tử bên trái. Các phần tử bị đụng đều ở chỉ số `< p1 < pn`, nên `n` không đổi chỗ. Sau đó đưa `n` về cuối bằng `n - 1 - pn` lần. Tổng `p1 + (n - 1 - pn)`.
- Nếu `p1 > pn`: đưa `1` về đầu bằng `p1` lần; trên đường đi `1` đổi chỗ với `n` đúng một lần, làm `n` chuyển từ `pn` sang `pn + 1`. Sau đó cần `n - 1 - (pn + 1)` lần. Tổng `p1 + n - 2 - pn = p1 + (n - 1 - pn) - 1`.

**Cận dưới (không thể ít hơn).** Mỗi lần đổi chỗ kề nhau dịch chuyển **mỗi phần tử tối đa một ô**. Để `1` tới chỉ số `0` cần ít nhất `p1` lần mà `1` bị dịch; để `n` tới chỉ số `n - 1` cần ít nhất `n - 1 - pn` lần mà `n` bị dịch. Một lần đổi chỗ có thể "phục vụ" cả hai số **chỉ khi** nó đổi chỗ chính `1` và `n` với nhau (đúng khi cả hai cùng bị dịch theo hướng có lợi: `n` sang phải và `1` sang trái, tức `1` đứng ngay sau `n`). Điều này **chỉ có thể xảy ra khi `1` đứng sau `n`** (`p1 > pn`), và **nhiều nhất một lần** (sau khi đổi, `n` đã đứng sau `1`, không còn "gặp lại" theo hướng có lợi). Vì vậy tổng số lần đổi chỗ `≥ p1 + (n - 1 - pn) - [p1 > pn]`.

Hai cận khớp nhau nên công thức đúng. Ta cũng kiểm chứng bằng **BFS vét cạn** ở mục 12.4.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)` (tối đa 50).

| Cách | Time | Space |
|---|---|---|
| 1. mô phỏng đổi chỗ | `O(n)` | `O(n)` (bản sao) |
| 2. công thức, hai `index` | `O(n)` | `O(1)` |
| 3. công thức, một lượt `enumerate` | `O(n)` | `O(1)` |
| 4. `pop` / `insert` / `append` | `O(n)` | `O(n)` (bản sao) |

Cận dưới `O(n)` vì phải tìm vị trí của `1` và `n` trong mảng chưa sắp. Cách 2 và 3 đạt cận đó với bộ nhớ phụ `O(1)`.

## 12. Bộ test

### 12.1. Ví dụ của đề

In [20]:
sols = [SolutionIndex(), SolutionOnePass(), SolutionPopInsert(), Solution()]
funcs = [semi_ordered_simulate, semi_ordered_index, semi_ordered_one_pass, semi_ordered_pop_insert] + [s.semiOrderedPermutation for s in sols]

for f in funcs:
    assert f([2, 1, 4, 3]) == 2
    assert f([2, 4, 1, 3]) == 3
    assert f([1, 3, 4, 2, 5]) == 0
print("Ví dụ của đề: tất cả cách đều đúng")

Ví dụ của đề: tất cả cách đều đúng


### 12.2. Edge case (chỉ dùng input hợp lệ: hoán vị của `1..n`)

- `n = 2`: `[1, 2]` → `0` (đã semi-ordered); `[2, 1]` → `1` (một lần đổi).
- `1` ở cuối và `n` ở đầu: `[3, 2, 1]` → `p1 = 2`, `pn = 0`: `2 + (3 - 1 - 0) - 1 = 3`.
- Đã đúng đầu, sai cuối: `[1, 3, 2]` → `1`.
- Sai đầu, đúng cuối: `[2, 1, 3]` → `1`.
- `1` ngay sau `n` (đổi chỗ chung): `[4, 1, 2, 3]` → `p1 = 1`, `pn = 0`: `1 + 3 - 1 = 3`.
- Đảo ngược hoàn toàn `n = 50`: `[50, 49, ..., 1]` → `49 + 49 - 1 = 97`.

In [21]:
cases = [
    ([1, 2], 0), ([2, 1], 1), ([3, 2, 1], 3), ([1, 3, 2], 1), ([2, 1, 3], 1),
    ([4, 1, 2, 3], 3), (list(range(50, 0, -1)), 97), (list(range(1, 51)), 0),
]
for f in funcs:
    for nums, want in cases:
        assert f(nums[:]) == want, (f, nums[:5], want)
print("Edge case OK:", len(cases), "trường hợp x", len(funcs), "cách")

# kiểm tra không làm hỏng mảng đầu vào
a = [2, 4, 1, 3]
b = a[:]
for s in sols:
    s.semiOrderedPermutation(a)
assert a == b
print("Không sửa mảng đầu vào")

Edge case OK: 8 trường hợp x 8 cách
Không sửa mảng đầu vào


### 12.3. Phản ví dụ cho cách làm sai

Hai cách sai hay gặp: quên trừ `1` khi `1` nằm sau `n`, và sắp xếp **cả mảng** (đếm số lần đổi chỗ kề nhau để sort hoàn toàn = số nghịch thế). Ta xác nhận chúng khác đáp án.

In [22]:
def sai_quen_tru(nums):
    n = len(nums)
    return nums.index(1) + (n - 1 - nums.index(n))

def sai_sap_xep_het(nums):
    # đếm nghịch thế = số lần đổi chỗ kề nhau tối thiểu để sắp xếp cả mảng
    return sum(1 for i in range(len(nums)) for j in range(i + 1, len(nums)) if nums[i] > nums[j])

for nums in ([2, 4, 1, 3], [1, 3, 2, 4]):
    print(nums, "đúng:", Solution().semiOrderedPermutation(nums), "| quên trừ:", sai_quen_tru(nums), "| sắp xếp hết:", sai_sap_xep_het(nums))
assert sai_quen_tru([2, 4, 1, 3]) != Solution().semiOrderedPermutation([2, 4, 1, 3])
assert sai_sap_xep_het([1, 3, 2, 4]) != Solution().semiOrderedPermutation([1, 3, 2, 4])

[2, 4, 1, 3] đúng: 3 | quên trừ: 4 | sắp xếp hết: 3
[1, 3, 2, 4] đúng: 0 | quên trừ: 0 | sắp xếp hết: 1


### 12.4. BFS vét cạn + đối chiếu ngẫu nhiên (có seed)

**Oracle độc lập:** tìm đường đi **ngắn nhất** trong "đồ thị hoán vị" (mỗi đỉnh là một hoán vị, mỗi cạnh là một lần đổi chỗ hai phần tử kề nhau) từ `nums` đến **một hoán vị semi-ordered bất kỳ** bằng **BFS**. BFS đảm bảo số bước ít nhất mà không cần giả định gì về công thức. Dùng cho `n ≤ 7` (`7! = 5040` trạng thái).

Sau đó kiểm tra các cách trên hoán vị ngẫu nhiên có `n` tới 50.

In [23]:
import random, itertools
from collections import deque

def oracle_bfs(nums):
    n = len(nums)
    start = tuple(nums)
    if start[0] == 1 and start[-1] == n:
        return 0
    seen = {start}
    dq = deque([(start, 0)])
    while dq:
        cur, d = dq.popleft()
        for i in range(n - 1):
            nxt = list(cur)
            nxt[i], nxt[i + 1] = nxt[i + 1], nxt[i]
            nxt = tuple(nxt)
            if nxt in seen:
                continue
            if nxt[0] == 1 and nxt[-1] == n:
                return d + 1
            seen.add(nxt)
            dq.append((nxt, d + 1))

# 1) mọi hoán vị với n = 2..5 (tổng 2 + 6 + 24 + 120 = 152 hoán vị)
total = 0
for n in range(2, 6):
    for perm in itertools.permutations(range(1, n + 1)):
        want = oracle_bfs(list(perm))
        for f in funcs:
            assert f(list(perm)) == want, (perm, f)
        total += 1
print("đã kiểm", total, "hoán vị (n = 2..5), 7 cách khớp BFS")

# 2) hoán vị ngẫu nhiên n = 6, 7 (có seed)
random.seed(2717)
for _ in range(40):
    n = random.choice([6, 7])
    nums = list(range(1, n + 1))
    random.shuffle(nums)
    want = oracle_bfs(nums)
    for f in funcs:
        assert f(nums[:]) == want, (nums, f)
print("OK: 40 hoán vị ngẫu nhiên n = 6, 7 khớp BFS")

đã kiểm 152 hoán vị (n = 2..5), 7 cách khớp BFS
OK: 40 hoán vị ngẫu nhiên n = 6, 7 khớp BFS


In [24]:
random.seed(27170)
for _ in range(2000):
    n = random.randint(2, 50)
    nums = list(range(1, n + 1))
    random.shuffle(nums)
    want = semi_ordered_simulate(nums)           # oracle = mô phỏng đúng luật đề (cách 1)
    for f in funcs:
        assert f(nums[:]) == want, (nums, f)
    assert 0 <= want <= 2 * (n - 1) - 1 or n == 1
print("OK: 2000 hoán vị ngẫu nhiên n tới 50, mọi cách khớp")

OK: 2000 hoán vị ngẫu nhiên n tới 50, mọi cách khớp


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên trừ `1` khi `1` đứng sau `n`

In [25]:
from typing import List

def sai_quen_tru_1(nums: List[int]) -> int:
    n = len(nums)
    return nums.index(1) + (n - 1 - nums.index(n))      # SAI: thiếu trừ khi chồng chéo

print("sai :", sai_quen_tru_1([2, 4, 1, 3]))                  # 4
print("đúng:", Solution().semiOrderedPermutation([2, 4, 1, 3]))   # 3
assert sai_quen_tru_1([2, 4, 1, 3]) == 4

sai : 4
đúng: 3


**Vì sao sai:** ví dụ 2 của đề chỉ ra rõ: `[2,4,1,3]` cần `3`, không phải `4`. Khi `1` ở sau `n`, một lần đổi chỗ giữa `1` và `n` vừa đưa `1` về đầu một ô vừa đưa `n` về cuối một ô; công thức cộng thẳng đã đếm lần đó hai lần.

### Lỗi 2: Tính khoảng cách tới cuối bằng `n - j` thay vì `n - 1 - j`

In [26]:
def sai_lech_mot(nums: List[int]) -> int:
    n = len(nums)
    i, j = nums.index(1), nums.index(n)
    ans = i + (n - j)                      # SAI: nên là n - 1 - j (chỉ số cuối là n - 1)
    if i > j:
        ans -= 1
    return ans

print("sai :", sai_lech_mot([1, 3, 4, 2, 5]))                # 1, nhưng đã semi-ordered nên đúng là 0
print("đúng:", Solution().semiOrderedPermutation([1, 3, 4, 2, 5]))
assert sai_lech_mot([1, 3, 4, 2, 5]) != 0

sai : 1
đúng: 0


**Vì sao sai:** chỉ số cuối là `n - 1`, nên số bước đưa phần tử ở chỉ số `j` tới cuối là `(n - 1) - j`. Nếu `n` đã nằm sẵn ở cuối (`j = n - 1`) thì cần `0` bước — công thức `n - j` cho `1` bước.

### Lỗi 3: Sắp xếp cả mảng / đếm mọi nghịch thế

Đề **không** yêu cầu sắp xếp toàn bộ mảng; chỉ cần đầu `= 1` và cuối `= n`. Các số giữa tuỳ ý.

In [27]:
def sai_dem_nghich_the(nums: List[int]) -> int:
    return sum(1 for i in range(len(nums)) for j in range(i + 1, len(nums)) if nums[i] > nums[j])

print("sai :", sai_dem_nghich_the([1, 3, 2, 4]))               # 1
print("đúng:", Solution().semiOrderedPermutation([1, 3, 2, 4]))   # 0
assert sai_dem_nghich_the([1, 3, 2, 4]) == 1

sai : 1
đúng: 0


**Vì sao sai:** `[1, 3, 2, 4]` chưa được sắp xếp (còn nghịch thế `3 > 2`) nhưng **đã semi-ordered** — đầu `1`, cuối `4`. Đáp án là `0`, còn đếm nghịch thế cho `1`.

### Lỗi 4: Xét điều kiện `i < j` thay vì `i > j` (hoặc dùng `>=`)

In [28]:
def sai_nguoc_dieu_kien(nums: List[int]) -> int:
    n = len(nums)
    i, j = nums.index(1), nums.index(n)
    ans = i + (n - 1 - j)
    if i < j:                              # SAI: ngược điều kiện
        ans -= 1
    return ans

print("sai :", sai_nguoc_dieu_kien([2, 1, 4, 3]))             # 1 (đúng phải là 2)
print("đúng:", Solution().semiOrderedPermutation([2, 1, 4, 3]))
assert sai_nguoc_dieu_kien([2, 1, 4, 3]) != Solution().semiOrderedPermutation([2, 1, 4, 3])

sai : 1
đúng: 2


**Vì sao sai:** chỉ khi `1` đứng **sau** `n` (`i > j`) hai số mới đi ngang qua nhau. Nếu `1` đứng trước `n` thì chúng không bao giờ gặp nhau, không có gì để trừ. Hãy tự thử hai ví dụ nhỏ để chắc chắn hướng của dấu `>`. (Vì `1 ≠ n` nên `i == j` không thể xảy ra, `>=` hay `>` cho cùng kết quả với input hợp lệ.)

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề cho **hoán vị** + "đổi chỗ kề nhau" ⇒ đáp án thường liên quan tới **khoảng cách chỉ số** (mỗi thao tác dịch mỗi phần tử một ô).
- Chỉ cần quan tâm các phần tử **có ràng buộc** (`1` và `n`); phần tử khác tuỳ ý ⇒ không cần sắp xếp.
- Khi hai "đường đi" có thể **cắt nhau**, luôn hỏi: "có bước nào được tính **hai lần** không?" — đó là nguồn của số hạng hiệu chỉnh.
- Tự kiểm công thức bằng **BFS** trên `n ≤ 6` — dễ viết và đảm bảo không sai sót.
- Với `list`: `index` (tìm vị trí), `pop(i)` + `insert(0, x)`/`append(x)` (chuyển phần tử) là bộ công cụ chuẩn.
- Tên biến tránh từ khoá (`in`, `is`, `not`, `and`...).

## 15. Biến thể của bài

1. **Sắp xếp hoàn toàn** bằng đổi chỗ kề nhau: số lần tối thiểu = **số nghịch thế** (đếm bằng merge sort `O(n log n)`).
2. **Đổi chỗ hai phần tử bất kỳ** (không cần kề nhau): đếm **chu trình** của hoán vị; số bước tối thiểu để sắp xếp = `n - số chu trình`.
3. **Đưa `1` về đầu và `n` về cuối** nhưng cho phép đổi **hai phần tử bất kỳ**: tối đa 2 thao tác.
4. **Hoán vị không đảm bảo là hoán vị `1..n`** (mảng bất kỳ): tìm min, max thay vì `1`, `n`.
5. **Đưa phần tử nhỏ nhất về đầu & lớn nhất về cuối bằng tối thiểu thao tác** với mảng có số lặp: xét vị trí xuất hiện đầu tiên của min và cuối cùng của max.
6. **LeetCode 2570 / 2656** là các bài mảng cùng loạt, và **LeetCode 1** (Two Sum) dùng `index` / `dict` tương tự.

In [29]:
def semi_ordered_generic(nums):
    """Biến thể 4: mảng bất kỳ có phần tử khác nhau; đưa MIN về đầu và MAX về cuối."""
    n = len(nums)
    lo, hi = nums.index(min(nums)), nums.index(max(nums))
    return lo + (n - 1 - hi) - (1 if lo > hi else 0)

print(semi_ordered_generic([5, 9, 2, 7]))        # min=2 ở 2, max=9 ở 1: 2 + 2 - 1 = 3

def swaps_to_sort_anywhere(nums):
    """Biến thể 2: đổi chỗ hai phần tử BẤT KỲ để sắp xếp hoán vị 1..n: n - số chu trình."""
    n = len(nums)
    seen = [False] * n
    cycles = 0
    for i in range(n):
        if not seen[i]:
            cycles += 1
            j = i
            while not seen[j]:
                seen[j] = True
                j = nums[j] - 1
    return n - cycles

print(swaps_to_sort_anywhere([2, 4, 1, 3]))      # 3 (một chu trình dài 4: n - 1 = 3)

3
3


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. mô phỏng đổi chỗ | thật sự đổi chỗ, đếm | `O(n)` | `O(n)` | hiểu đề, làm oracle |
| 2. công thức, hai `index` | `i + (n-1-j) - [i>j]` | `O(n)` | `O(1)` | **cách nộp bài khuyến nghị** |
| 3. công thức, một lượt | tìm `i`, `j` trong 1 vòng | `O(n)` | `O(1)` | khi muốn đọc mảng đúng một lần |
| 4. `pop` / `insert` / `append` | nhấc phần tử ra đặt vào chỗ mới | `O(n)` | `O(n)` | giải thích vì sao có số hạng trừ 1 |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `[3, 1, 2].index(1)`, `5 - (4 > 6)`, `a = [1, 2, 3]; a.insert(0, a.pop(2))` ra gì, và `True + 2`.

**Bài 2 (tính tay).** Với `nums = [3, 4, 1, 2]` (`n = 4`), tính số thao tác bằng công thức. Sau đó viết ra từng lần đổi chỗ để kiểm tra.

**Bài 3 (điền TODO).** Hoàn thành `semi_ordered_todo(nums)` bằng công thức.

In [30]:
def semi_ordered_todo(nums):
    n = len(nums)
    # TODO 1: i = vị trí của 1, j = vị trí của n
    # TODO 2: ans = i + (n - 1 - j)
    # TODO 3: nếu i > j thì trừ 1
    # TODO 4: return ans
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `semi_ordered_one_pass2(nums)` tìm hai vị trí **trong một vòng `enumerate`** mà không nhìn Cách 3. Kiểm tra với ba ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `inversions(nums)` đếm **số nghịch thế** (số cặp `i < j` với `nums[i] > nums[j]`) — đó là số lần đổi chỗ kề nhau tối thiểu để **sắp xếp hoàn toàn**. (b) Viết `min_swaps_to_ends(nums)` cho mảng có phần tử **khác nhau bất kỳ** (không nhất thiết `1..n`): đưa **min về đầu và max về cuối** bằng đổi chỗ kề nhau, trả số thao tác.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `index(1)` là `1`; `5 - False = 5`; `[3, 1, 2]`; `True + 2 = 3`. Bài 2: `1` ở chỉ số 2, `4` ở chỉ số 1.
- **Mức 2 (vừa):** Bài 3: `i = nums.index(1)`, `j = nums.index(n)`, công thức rồi `if i > j: ans -= 1`. Bài 4: `for i, x in enumerate(nums)`; `if x == 1: p1 = i` / `elif x == n: pn = i`.
- **Mức 3 (sâu):** Bài 5(a): hai vòng lặp lồng nhau cho `n ≤ 50` là đủ. Bài 5(b): thay `1` bằng `min(nums)`, `n` bằng `max(nums)`; chỉ số `lo`, `hi` rồi dùng công thức cũ.

## 19. Đáp án tham khảo cho bài tập

In [31]:
# Bài 1
assert [3, 1, 2].index(1) == 1
assert 5 - (4 > 6) == 5
a = [1, 2, 3]
a.insert(0, a.pop(2))
assert a == [3, 1, 2]
assert True + 2 == 3

# Bài 2: [3,4,1,2]: i = 2, j = 1, n = 4: 2 + (4 - 1 - 1) - 1 = 3
assert Solution().semiOrderedPermutation([3, 4, 1, 2]) == 3
b = [3, 4, 1, 2]
cnt = 0
for idx in (1, 0, 2):             # đổi (4,1) -> [3,1,4,2]; đổi (3,1) -> [1,3,4,2]; đổi (4,2) -> [1,3,2,4]
    b[idx], b[idx + 1] = b[idx + 1], b[idx]
    cnt += 1
assert b[0] == 1 and b[-1] == 4 and cnt == 3

# Bài 3
def semi_ordered_todo_ans(nums):
    n = len(nums)
    i, j = nums.index(1), nums.index(n)
    ans = i + (n - 1 - j)
    if i > j:
        ans -= 1
    return ans

assert semi_ordered_todo_ans([2, 1, 4, 3]) == 2
assert semi_ordered_todo_ans([2, 4, 1, 3]) == 3
assert semi_ordered_todo_ans([1, 3, 4, 2, 5]) == 0

# Bài 4
def semi_ordered_one_pass2(nums):
    n = len(nums)
    p1 = pn = -1
    for i, x in enumerate(nums):
        if x == 1:
            p1 = i
        elif x == n:
            pn = i
    return p1 + (n - 1 - pn) - (1 if p1 > pn else 0)

assert semi_ordered_one_pass2([2, 1, 4, 3]) == 2
assert semi_ordered_one_pass2([2, 4, 1, 3]) == 3
assert semi_ordered_one_pass2([1, 3, 4, 2, 5]) == 0

# Bài 5
def inversions(nums):
    return sum(1 for i in range(len(nums)) for j in range(i + 1, len(nums)) if nums[i] > nums[j])

assert inversions([2, 1, 4, 3]) == 2
assert inversions([3, 2, 1]) == 3
assert inversions([1, 2, 3]) == 0

def min_swaps_to_ends(nums):
    n = len(nums)
    lo, hi = nums.index(min(nums)), nums.index(max(nums))
    return lo + (n - 1 - hi) - (1 if lo > hi else 0)

assert min_swaps_to_ends([5, 9, 2, 7]) == 3
assert min_swaps_to_ends([2, 1, 4, 3]) == 2
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (vị trí của `1` và `n`, trừ `1` nếu chúng "bắt chéo")

```python
from typing import List

class Solution:
    def semiOrderedPermutation(self, nums: List[int]) -> int:
        n = len(nums)
        i = nums.index(1)
        j = nums.index(n)
        ans = i + (n - 1 - j)
        if i > j:
            ans -= 1
        return ans
```

**Checklist**

- [ ] Chỉ cần đưa `1` về đầu và `n` về cuối (không sắp xếp cả mảng)?
- [ ] Khoảng cách của `1`: `i`; khoảng cách của `n`: `n - 1 - j` (có `- 1`)?
- [ ] Trừ `1` **khi và chỉ khi** `1` đứng **sau** `n` (`i > j`)?
- [ ] Không đặt tên biến là `in` (từ khoá)?
- [ ] Đã thử `[1, 2]`, `[2, 1]`, `[3, 2, 1]`, `[4, 1, 2, 3]` và hoán vị ngược `n = 50`?
- [ ] Đã đối chiếu với BFS / mô phỏng trên input nhỏ?
- [ ] Signature đúng `semiOrderedPermutation(self, nums: List[int]) -> int` và có `from typing import List`?